# Compact Neural-DTB oscillatory game

This notebook contains one complete experiment for one parameter choice. It derives the equilibrium equations and stability conditions, defines the game, samples the particles, computes a standard RK4 reference, pushes the same particles with Neural-DTB, and makes only three diagnostic figures.

The notebook imports the existing DTB core, but it does not call the repository's sweep runner or generate tables and output folders. In Google Colab, first choose **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom.

In [ ]:
# Colab setup
import os, pathlib, subprocess, sys

REPO_DIR = pathlib.Path("/content/dtb-colab-experiments")
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "-q", "--depth", "1",
        "--branch", "codex/game-dynamics-dtb",
        "https://github.com/sun-mengwei/dtb-colab-experiments.git",
        str(REPO_DIR),
    ], check=True)
os.chdir(REPO_DIR / "dtb_game_dynamics_unnormalized")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## Game and numerical setup

For $x=(x_1,x_2)$, the potential and game velocity are

$$\Phi(x)=-\frac{\lambda}{2}(x_1^2+x_2^2)-\frac{\gamma}{2}(x_1-x_2)^2+\frac{\varepsilon}{\omega}[\cos(\omega x_1)+\cos(\omega x_2)],$$

$$b(x)=\nabla\Phi(x)=\begin{bmatrix}-\lambda x_1-\gamma(x_1-x_2)-\varepsilon\sin(\omega x_1)\\-\lambda x_2-\gamma(x_2-x_1)-\varepsilon\sin(\omega x_2)\end{bmatrix}.$$

The diffusion matrix is zero, so the DTB target velocity is exactly $b(x)$. The standard solver and DTB start from the same particles sampled uniformly from $[-1,1]^2$.

## Analytical equilibrium equations

An equilibrium $x^*=(x_1^*,x_2^*)$ satisfies $b(x^*)=0$, or

$$ (\lambda+\gamma)x_1^*-\gamma x_2^*+\varepsilon\sin(\omega x_1^*)=0, \qquad (\lambda+\gamma)x_2^*-\gamma x_1^*+\varepsilon\sin(\omega x_2^*)=0. $$

Use the mean and difference coordinates

$$m=\frac{x_1+x_2}{2}, \qquad d=\frac{x_1-x_2}{2}.$$

Adding and subtracting the two equilibrium equations gives the exact characterization

$$\boxed{\lambda m+\varepsilon\sin(\omega m)\cos(\omega d)=0},$$

$$\boxed{(\lambda+2\gamma)d+\varepsilon\cos(\omega m)\sin(\omega d)=0}.$$

Thus the symmetric equilibria $(s,s)$ are the roots of

$$\lambda s+\varepsilon\sin(\omega s)=0,$$

and the antisymmetric equilibria $(s,-s)$ are the roots of

$$(\lambda+2\gamma)s+\varepsilon\sin(\omega s)=0.$$

Because these are transcendental equations, nonzero roots generally have no elementary closed form; the two boxed equations are the analytical solution condition, and individual nonzero roots must be evaluated numerically. The origin is always an exact equilibrium.

The game Jacobian is

$$J_b(x)=\begin{bmatrix}-\lambda-\gamma-\varepsilon\omega\cos(\omega x_1)&\gamma\\\gamma&-\lambda-\gamma-\varepsilon\omega\cos(\omega x_2)\end{bmatrix}.$$

Since $b=\nabla\Phi$, an isolated equilibrium is locally asymptotically stable when this symmetric matrix is negative definite. At the origin its eigenvalues are

$$\mu_{\mathrm{mean}}=-\lambda-\varepsilon\omega, \qquad \mu_{\mathrm{difference}}=-\lambda-2\gamma-\varepsilon\omega.$$

Therefore the origin is stable exactly when both eigenvalues are negative. For the parameters below they are approximately $-6.7832$ and $-7.1832$, so at least one stable equilibrium exists.

In [ ]:
import math
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import torch

from game_dtb.algorithm import DTBConfig, NeuralDTBGameDynamics
from game_dtb.models import TangentMLP
from game_dtb.state import ParticleState

# One game only. Change these values later to study another case.
LAMBDA = 0.5
GAMMA = 0.2
EPSILON = 0.5
OMEGA = 4.0 * math.pi

PARTICLES = 128
STEPS = 1000
STEP_SIZE = 0.001       # 1000 steps keep the final time T = 1
RK4_SUBSTEPS = 1        # RK4 already uses the small 0.001 step
SEED = 2026
MODEL_SEED = 91
WIDTH = 12
DEPTH = 2
BASIS_SIZE = 24
SVD_RTOL = 1e-3

# Periodically fit the NN to the accumulated DTB tangent target.
# Set REFIT_INTERVAL = 0 to recover the fixed-network version.
REFIT_INTERVAL = 100
REFIT_OPTIMIZER_STEPS = 300
REFIT_LEARNING_RATE = 1e-3
REFIT_SAMPLES = 1024
REFIT_BATCH_SIZE = 256
REFIT_TEST_SAMPLES = 512

DTYPE = torch.float32
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.set_float32_matmul_precision("high")

pprint({
    "lambda": LAMBDA, "gamma": GAMMA, "epsilon": EPSILON,
    "omega": f"{OMEGA / math.pi:g} pi",
    "particles": PARTICLES, "steps": STEPS,
    "step_size": STEP_SIZE, "final_time": STEPS * STEP_SIZE,
    "network": f"TangentMLP(width={WIDTH}, depth={DEPTH})",
    "basis_size": BASIS_SIZE, "svd_rtol": SVD_RTOL,
    "refit_interval": REFIT_INTERVAL,
    "planned_refits": STEPS // REFIT_INTERVAL if REFIT_INTERVAL else 0,
    "device": str(DEVICE),
})
if DEVICE.type != "cuda":
    print("GPU not detected. In Colab choose Runtime > Change runtime type > T4 GPU.")

## Functions used in this experiment

These are the complete experiment-specific functions. The imported DTB class performs the tangent projection and particle update. There are no sweep, report, dimension-check, or test-suite cells here.

In [ ]:
def potential(x):
    x1, x2 = x.unbind(dim=-1)
    return (
        -0.5 * LAMBDA * (x1.square() + x2.square())
        -0.5 * GAMMA * (x1 - x2).square()
        +(EPSILON / OMEGA) * (torch.cos(OMEGA * x1) + torch.cos(OMEGA * x2))
    )


def velocity(x):
    x1, x2 = x.unbind(dim=-1)
    v1 = -LAMBDA * x1 - GAMMA * (x1 - x2) - EPSILON * torch.sin(OMEGA * x1)
    v2 = -LAMBDA * x2 - GAMMA * (x2 - x1) - EPSILON * torch.sin(OMEGA * x2)
    return torch.stack((v1, v2), dim=-1)


def make_initial_state():
    generator = torch.Generator().manual_seed(SEED)
    labels = 2.0 * torch.rand(PARTICLES, 2, generator=generator, dtype=DTYPE) - 1.0
    labels = labels.to(DEVICE)
    return ParticleState(
        particles=labels.clone(),
        log_density=torch.full((PARTICLES,), -math.log(4.0), device=DEVICE, dtype=DTYPE),
        score=torch.zeros_like(labels),
        labels=labels,
    )


def solve_with_rk4(initial_particles):
    x = initial_particles.clone()
    history = [x.cpu().numpy().copy()]
    dt = STEP_SIZE / RK4_SUBSTEPS
    with torch.no_grad():
        for _ in range(STEPS):
            for _ in range(RK4_SUBSTEPS):
                k1 = velocity(x)
                k2 = velocity(x + 0.5 * dt * k1)
                k3 = velocity(x + 0.5 * dt * k2)
                k4 = velocity(x + dt * k3)
                x = x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
            history.append(x.cpu().numpy().copy())
    return np.stack(history)


def solve_with_dtb(initial_state):
    torch.manual_seed(MODEL_SEED)
    model = TangentMLP(
        dim=2, width=WIDTH, depth=DEPTH, activation="tanh", dtype=DTYPE
    ).to(DEVICE)
    config = DTBConfig(
        step_size=STEP_SIZE, basis_size=BASIS_SIZE, svd_rtol=SVD_RTOL,
        jacobian_chunk_size=PARTICLES, derivative_chunk_size=32, seed=SEED,
        refit_interval=REFIT_INTERVAL,
        refit_optimizer_steps=REFIT_OPTIMIZER_STEPS,
        refit_learning_rate=REFIT_LEARNING_RATE,
        refit_batch_size=REFIT_BATCH_SIZE,
        refit_samples=REFIT_SAMPLES,
        refit_test_samples=REFIT_TEST_SAMPLES,
    )
    method = NeuralDTBGameDynamics(
        model=model, drift=velocity,
        diffusion=torch.zeros(2, 2, device=DEVICE, dtype=DTYPE),
        config=config,
        # With no separate sampler, refitting samples the particle-label dataset.
        reference_sampler=None,
    )

    state = initial_state
    history = [state.particles.cpu().numpy().copy()]
    residuals = []
    refit_steps = []
    for step in range(STEPS):
        result = method.step(state, step)
        state = result.state
        history.append(state.particles.cpu().numpy().copy())
        residuals.append(result.diagnostics.relative_residual)
        if result.refit_performed:
            refit_steps.append(step + 1)
    return np.stack(history), np.asarray(residuals), refit_steps


def origin_stability():
    # b(0,0)=0. Negative Jacobian eigenvalues imply local asymptotic stability.
    diagonal = -LAMBDA - GAMMA - EPSILON * OMEGA
    jacobian = torch.tensor(
        [[diagonal, GAMMA], [GAMMA, diagonal]], dtype=torch.float64
    )
    eigenvalues = torch.linalg.eigvalsh(jacobian).numpy()
    return bool(eigenvalues.max() < 0.0), eigenvalues


def plot_particle_snapshots(history, title):
    indices = np.linspace(0, STEPS, 5, dtype=int)
    fig, axes = plt.subplots(1, len(indices), figsize=(15, 3), sharex=True, sharey=True)
    for axis, index in zip(axes, indices):
        points = history[index]
        axis.scatter(points[:, 0], points[:, 1], s=10, alpha=0.65)
        axis.set_title(f"t = {index * STEP_SIZE:.2f}")
        axis.set_xlim(-1.1, 1.1)
        axis.set_ylim(-1.1, 1.1)
        axis.set_aspect("equal")
        axis.grid(alpha=0.2)
    axes[0].set_ylabel(r"$x_2$")
    for axis in axes:
        axis.set_xlabel(r"$x_1$")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## Run the two solvers

The two methods below receive identical initial particles. For this parameter set, the origin alone is enough to establish the existence of a stable solution: it is always an equilibrium, and the code checks the two eigenvalues of its game Jacobian.

The DTB loop holds one tangent basis for each 100-step block, accumulates its update, and then refits all trainable MLP parameters to that accumulated tangent target using samples drawn from the initial particle-label dataset. This is the DTB reset/refit operation—not direct supervised fitting to $b(x)$. A 1,000-step DTB run is computationally substantial, so enable a GPU in Colab before running this cell.

In [ ]:
initial_state = make_initial_state()
ode_history = solve_with_rk4(initial_state.particles)
dtb_history, projection_residual, refit_steps = solve_with_dtb(initial_state)
times = np.arange(STEPS + 1) * STEP_SIZE

stable, eigenvalues = origin_stability()
print(
    f"Stable solution exists for gamma={GAMMA:g}, lambda={LAMBDA:g}: {stable}. "
    f"Origin Jacobian eigenvalues: {eigenvalues}. "
    f"NN refits completed: {len(refit_steps)} at steps {refit_steps}."
)

## Projection residual over time

At each DTB step, this is $\|J\alpha-b\|_2/\|b\|_2$: how well the selected neural tangent directions represent the game velocity at the current particles.

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.semilogy(times[1:], np.maximum(projection_residual, 1e-12), label="DTB residual")
for index, step in enumerate(refit_steps):
    plt.axvline(times[step], color="tab:orange", alpha=0.25,
                label="NN refit" if index == 0 else None)
plt.xlabel("time")
plt.ylabel("relative projection residual")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Standard ODE solution over time

In [ ]:
plot_particle_snapshots(ode_history, "Standard RK4 particle solution")

## Neural-DTB solution over time

In [ ]:
plot_particle_snapshots(dtb_history, "Neural-DTB particle solution")